In [68]:
import pandas as pd
from pandas.plotting import scatter_matrix
import mglearn

In [69]:
dataframe = pd.read_csv("data/train.csv")
dataframe.head()

,id,breath_id,R,C,time_step,u_in,u_out,pressure
0,1,1,20,50,0.000000,0.083334,0,5.837492
1,2,1,20,50,0.033652,18.383041,0,5.907794
2,3,1,20,50,0.067514,22.509278,0,7.876254
3,4,1,20,50,0.101542,22.808822,0,11.742872
4,5,1,20,50,0.135756,25.355850,0,12.234987


In [70]:
parameters = dataframe.columns
df_data = dataframe.drop(columns=['pressure','id'])
df_labels = dataframe.drop(columns=list(parameters)[:len(parameters)-1])
df_data.head(), df_labels.head()


(   breath_id   R   C  time_step       u_in  u_out
 0          1  20  50   0.000000   0.083334      0
 1          1  20  50   0.033652  18.383041      0
 2          1  20  50   0.067514  22.509278      0
 3          1  20  50   0.101542  22.808822      0
 4          1  20  50   0.135756  25.355850      0,
     pressure
 0   5.837492
 1   5.907794
 2   7.876254
 3  11.742872
 4  12.234987)

In [71]:
# grr = scatter_matrix(dataframe[:75],c=dataframe['pressure'][:75], figsize=(20,20), marker='o', hist_kwds={'bins': 20}, s=60, alpha=.8, cmap=mglearn.cm3)

In [72]:
labels = df_labels.to_numpy()
data = df_data.to_numpy()

In [73]:
# scatter_matrix(df_data.drop(columns=['id'])[:100], figsize=(15,15), c=labels[:100])

### Scaling data

In [74]:
import sklearn
from sklearn.preprocessing import StandardScaler

scalar = StandardScaler()
scalar.fit(df_data.drop(columns='breath_id'))
data_scaled = scalar.transform(df_data.drop(columns='breath_id'))

In [75]:
print(data_scaled.shape)

(6036000, 5)


### Principle components analysis

In [76]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
pca.fit(data_scaled)

data_pca = pca.transform(data_scaled)
print(data_scaled.shape)
print(data_pca.shape)
print(labels)

(6036000, 5)
(6036000, 2)
[[5.83749171]
 [5.90779385]
 [7.87625392]
 ...
 [3.79872949]
 [4.07993807]
 [3.86903163]]


In [77]:
print(data_pca[:10, 0])
print(labels[:10].squeeze())

[-1.73484917 -2.32077394 -2.43158027 -2.41402105 -2.47162382 -2.50789931
 -2.47588074 -2.43757967 -2.44538091 -2.4328659 ]
[ 5.83749171  5.90779385  7.87625392 11.74287192 12.23498694 12.86770625
 14.69556203 15.8906985  15.53918778 15.75009421]


In [78]:
# import matplotlib.pyplot as plt

# plt.figure(figsize=(10,10))

# mglearn.discrete_scatter(data_pca[:, 0], data_pca[:, 1], labels[:].squeeze())
# plt.gca().set_aspect("equal")


### Splitting data

In [79]:
from sklearn.model_selection import train_test_split

In [80]:
X_train, X_test, y_train, y_test = train_test_split(data_scaled, labels, train_size=0.8, random_state=42)

In [81]:
print(X_train.shape, y_test.shape)

(4828800, 5) (1207200, 1)


In [82]:
y_train, y_test = y_train.squeeze(), y_test.squeeze()

In [83]:
X_train_df = pd.DataFrame(X_train)
X_train_df.head()

,0,1,2,3,4
0,1.171893,-0.354513,0.726135,-0.225737,0.782135
1,1.171893,1.394522,0.506360,-0.302757,0.782135
2,1.171893,1.394522,-0.131811,-0.544978,0.782135
3,1.171893,-0.937525,-0.398159,-0.544978,0.782135
4,1.171893,-0.354513,-1.581420,-0.544978,-1.278552


### Tuning gradient boosting regressor hyperparameters

In [84]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV

param_grid = {
    "max_depth": [0,1,2,3,4,5,6,7,8,9,10],
    "learning_rate": [0, 0.1, 0.001, 0.0001]
}

gbr = GradientBoostingRegressor()
grid_search = GridSearchCV(estimator=gbr, param_grid=param_grid, cv=5, n_jobs=-1)



In [85]:
grid_search.fit(X_train[:1000], y_train[:1000])

c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py:489: FitFailedWarning: 
20 fits failed out of a total of 220.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
20 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py", line 1393, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",GradientBoostingRegressor()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0, 0.1, ...], 'max_depth': [0, 1, ...]}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of information print

In [86]:
best_model = grid_search.best_estimator_

In [87]:
best_model.score(X_test,y_test)

0.6701362087789637

In [88]:
print(grid_search.best_params_)

{'learning_rate': 0.1, 'max_depth': 3}


In [89]:
params = grid_search.best_estimator_.get_params()
params

{'alpha': 0.9,
 'ccp_alpha': 0.0,
 'criterion': 'deprecated',
 'init': None,
 'learning_rate': 0.1,
 'loss': 'squared_error',
 'max_depth': 3,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_iter_no_change': None,
 'random_state': None,
 'subsample': 1.0,
 'tol': 0.0001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

In [130]:
model_1 = GradientBoostingRegressor(
    alpha=0.9,
    ccp_alpha=0.0,
    criterion='deprecated',
    init=None,
    learning_rate=0.1,
    loss='squared_error',
    max_depth=3,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    min_samples_leaf=1,
    min_samples_split=2,
    min_weight_fraction_leaf=0.0,
    n_estimators=100,
    n_iter_no_change=None,
    random_state=None,
    subsample=1,
    tol=0.0001,
    validation_fraction=0.1,
    verbose=0,
    warm_start=False
)

In [138]:
model_1.fit(X_train[:500000],y_train[:500000])

,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"init init

In [92]:
scalar.fit(X_test)
X_test_scaled = scalar.transform(X_test)
model_1.score(X_test,y_test), X_test_scaled.shape

(0.7000583468739008, (1207200, 5))

In [93]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

In [94]:
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    acc = (correct / len(y_pred)) * 100
    return acc

In [95]:
print(X_test.shape)

(1207200, 5)


In [96]:
class BreathModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Linear(in_features=5, out_features=10),
            nn.ReLU(),
            nn.Linear(in_features=10, out_features=10),
            nn.ReLU(),
            nn.Linear(in_features=10, out_features=1)
        )
        
    def forward(self, x):
        return self.layer1(x)

In [97]:
model_0 = BreathModel()

In [98]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(params=model_0.parameters(),
                             lr=0.001)

In [99]:
BATCH_SIZE = 32

train_data_dataloader = DataLoader(
    dataset=torch.tensor(X_train).type(torch.float32),
    batch_size=BATCH_SIZE,
    shuffle=False
)
train_labels_dataloader = DataLoader(
    dataset=torch.tensor(y_train).type(torch.float32),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_data_dataloader = DataLoader(
    dataset=torch.tensor(X_test).type(torch.float32),
    batch_size=BATCH_SIZE,
    shuffle=False
)
test_labels_dataloader = DataLoader(
    dataset=torch.tensor(y_test).type(torch.float32),
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [100]:
print(next(iter(train_data_dataloader)).shape)

torch.Size([32, 5])


In [101]:
# for batches in train_data_dataloader:
#     print(batches.shape)

In [102]:
model_0(next(iter(test_data_dataloader))[:32]).shape

torch.Size([32, 1])

In [103]:
next(iter(train_labels_dataloader))[:32].unsqueeze(dim=1).shape

torch.Size([32, 1])

In [104]:
epochs = 3


for epoch in range(epochs):
    print(f"Epoch: {epoch}")
    train_loss = 0
    
    for batches, (X, y) in enumerate(zip(train_data_dataloader, train_labels_dataloader)):
        model_0.train()
        
        y_pred = model_0(X)
        
        if y_pred.ndim == y.unsqueeze(dim=1).ndim:
            loss = loss_fn(y_pred, y.unsqueeze(dim=1))
        else:
            raise ValueError("UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size. return F.mse_loss(input, target, reduction=self.reduction)")
        train_loss += loss
        
        optimizer.zero_grad()
        
        loss.backward()
        
        optimizer.step()
        
        if batches%32 == 0:
            print(f"looked at: {batches/32} samples/{len(train_data_dataloader)/32} of epoch{epoch}/{epochs}")
        
    train_loss /= len(train_data_dataloader)
    
    test_loss, test_acc = 0, 0
    model_0.eval()
    with torch.inference_mode():
        for X_batch,y_batch in zip(test_data_dataloader, test_labels_dataloader):
            test_pred = model_0(X_batch)

            test_loss += loss_fn(test_pred, y_batch.unsqueeze(dim=1))

            # 3. Calculate accuracy
            test_acc += accuracy_fn(y_true=y_batch, y_pred=test_pred.argmax(dim=1))
        
        # Calculate the test loss average per batch
        test_loss /= len(test_data_dataloader)

        # Calculate the test acc average per batch
        test_acc /= len(test_data_dataloader)
    
    # print out what's happening
    print(f"\nTrain loss: {train_loss:.4f} | Test loss: {test_loss:.4f}, Test acc: {test_acc:.4f}")
        

Epoch: 0
looked at: 0.0 samples
looked at: 1.0 samples
looked at: 2.0 samples
looked at: 3.0 samples
looked at: 4.0 samples
looked at: 5.0 samples
looked at: 6.0 samples
looked at: 7.0 samples
looked at: 8.0 samples
looked at: 9.0 samples
looked at: 10.0 samples
looked at: 11.0 samples
looked at: 12.0 samples
looked at: 13.0 samples
looked at: 14.0 samples
looked at: 15.0 samples
looked at: 16.0 samples
looked at: 17.0 samples
looked at: 18.0 samples
looked at: 19.0 samples
looked at: 20.0 samples
looked at: 21.0 samples
looked at: 22.0 samples
looked at: 23.0 samples
looked at: 24.0 samples
looked at: 25.0 samples
looked at: 26.0 samples
looked at: 27.0 samples
looked at: 28.0 samples
looked at: 29.0 samples
looked at: 30.0 samples
looked at: 31.0 samples
looked at: 32.0 samples
looked at: 33.0 samples
looked at: 34.0 samples
looked at: 35.0 samples
looked at: 36.0 samples
looked at: 37.0 samples
looked at: 38.0 samples
looked at: 39.0 samples
looked at: 40.0 samples
looked at: 41.0 s

In [105]:
model_0(next(iter(test_data_dataloader))[:10]), next(iter(test_labels_dataloader))[:10]

(tensor([[ 4.9415],
         [ 8.1006],
         [20.1806],
         [ 6.9369],
         [20.1941],
         [19.3690],
         [12.7379],
         [16.1028],
         [ 6.9224],
         [20.0614]], grad_fn=<AddmmBackward0>),
 tensor([ 4.5018,  8.2278, 14.2737,  6.1187, 18.5622, 17.8592, 10.4774, 14.2034,
          6.8217, 34.8020]))

In [111]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

In [128]:
pred = model_0(next(iter(test_data_dataloader))).squeeze().detach().numpy()
true_test_labels = next(iter(test_labels_dataloader)).detach().numpy()
mse = mean_squared_error(true_test_labels, pred)
mae = mean_absolute_error(true_test_labels, pred)
r2 = r2_score(true_test_labels, pred)
print(f"(MAE): {mae}: An MAE of {mae} indicates that, on average, the model’s predictions deviate from the actual values by approximately {mae} units.")
print(f"(MSE): {mse}: An MSE of {mse} shows that the average of the squared prediction errors is {mse}, meaning the model incurs some large errors that are heavily penalized.")
print(f"(R²): {r2}: An R² value of {r2} indicates that the model explains approximately {r2*100:.2f}% of the variance in the target variable, reflecting moderate predictive capability.")
print(f"(RMSE): : An RMSE of {np.sqrt(mse)} suggests that the model’s predictions typically differ from the actual values by about {np.sqrt(mse)} units, in the same scale as the target variable.")

(MAE): 2.8370800018310547: An MAE of 2.8370800018310547 indicates that, on average, the model’s predictions deviate from the actual values by approximately 2.8370800018310547 units.
(MSE): 25.912456512451172: An MSE of 25.912456512451172 shows that the average of the squared prediction errors is 25.912456512451172, meaning the model incurs some large errors that are heavily penalized.
(R²): 0.6907756924629211: An R² value of 0.6907756924629211 indicates that the model explains approximately 69.08% of the variance in the target variable, reflecting moderate predictive capability.
(RMSE): : An RMSE of 5.090427930189286 suggests that the model’s predictions typically differ from the actual values by about 5.090427930189286 units, in the same scale as the target variable.


#### model_1: tree-based model
#### model_0 : neural network

In [143]:
model_1.predict(X_test_scaled)[:5], y_test[:5]

(array([ 5.63253803,  6.96513598, 22.09172986,  6.213493  , 21.33522729]),
 array([ 4.50175094,  8.22776465, 14.27374916,  6.11870029, 18.56218003]))

In [139]:
tree_based_model_prediction = model_1.predict(X_test_scaled)
tree_mse = mean_squared_error(y_test, tree_based_model_prediction)
tree_mae = mean_absolute_error(y_test, tree_based_model_prediction)
tree_r2 = r2_score(y_test, tree_based_model_prediction)
print(tree_mse, tree_mae, tree_r2)

22.011399985787712 2.468850262450004 0.6653838773395785


In [141]:
tree_performance = [tree_mae, tree_mse, f"{tree_r2*100:.2f}%", np.sqrt(tree_mse)]
nn_performance = [mae, mse, f"{r2*100:.2f}%", np.sqrt(mse)]
scoring_system = ["MAE", "MSE", "R2", "SMSE"]
comparing_data = {
    "Tree-based model": tree_performance,
    "Neural Network": nn_performance
}

comparing_df = pd.DataFrame(comparing_data, index=scoring_system)

comparing_df

,Tree-based model,Neural Network
MAE,2.46885,2.83708
MSE,22.0114,25.912457
R2,66.54%,69.08%
SMSE,4.691631,5.090428
